# Lab Orpheus (Colab)

Open-weight: `orpheus-speech` + **vllm==0.7.3**. Not the paid `orpheus-tts` API.

## If you see `numpy.dtype size changed`
This is **not** fixed by pip alone in the same session. You must:

1. Run the **INSTALL-ONLY** cell (no `import vllm` in that cell)
2. **Runtime → Restart session** (or Disconnect and delete runtime)
3. Run **IMPORT-CHECK** cell only
4. Then prepare/render

## Official path
https://github.com/canopyai/Orpheus-TTS  
`pip install orpheus-speech` then `pip install vllm==0.7.3`  
Official Colab: https://colab.research.google.com/drive/1KhXT56UePPUHhqitJNUxq63k-pQomz3N


In [ ]:
REPO_URL = "https://github.com/phamtu2x5/ielts-script2audio.git"
BRANCH = "main"
WORKDIR = "/content/ielts-script2audio"


In [ ]:
import os
from pathlib import Path

if not Path(WORKDIR).exists():
    !git clone --branch {BRANCH} {REPO_URL} {WORKDIR}
else:
    %cd {WORKDIR}
    !git pull origin {BRANCH}
%cd {WORKDIR}
assert Path("pyproject.toml").is_file()
print("CWD", os.getcwd())


## A. INSTALL ONLY (do not import vllm/Orpheus in this cell)

After this cell finishes successfully, do **Runtime → Restart session**, then skip back only to IMPORT-CHECK.


In [ ]:
# ========== A. INSTALL ONLY — no import vllm here ==========
import sys
import subprocess

def pipi(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print(">", " ".join(cmd), flush=True)
    subprocess.check_call(cmd)

print("python", sys.version)

# Light project deps
pipi("-e", ".[dev]")

# Drop paid API client if any
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "orpheus-tts"])

# Upstream Orpheus local stack
# https://github.com/canopyai/Orpheus-TTS
pipi("orpheus-speech")
pipi("vllm==0.7.3")

# Pin stack that Colab churns (order: after orpheus/vllm so we win)
pipi("--force-reinstall", "numpy==1.26.4")
pipi("--force-reinstall", "pillow==10.4.0")
pipi("soundfile")

# Show what pip thinks is installed (not what is loaded in memory)
for pkg in ("numpy", "vllm", "torch", "pillow", "orpheus-speech"):
    subprocess.call([sys.executable, "-m", "pip", "show", pkg])

print(
    """
============================================================
STOP HERE — REQUIRED
1) Menu: Runtime → Restart session
2) After restart: run clone/pull cell if needed (WORKDIR)
3) Run the next cell: IMPORT-CHECK only (no reinstall unless import fails)
4) Do NOT import vllm in the same session right after force-reinstall
============================================================
"""
)


## B. IMPORT-CHECK (only after Restart)

If this fails with `dtype size changed` again: **Disconnect and delete runtime** → new GPU → A then Restart → B.


In [ ]:
# ========== B. IMPORT-CHECK (after Restart) ==========
import sys
print("python", sys.version)

import numpy as np
print("numpy", np.__version__)

import torch
print("torch", torch.__version__, "cuda", torch.version.cuda)
print("cuda_available", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU: Runtime → Change runtime type → GPU, then delete runtime if needed")
print("device", torch.cuda.get_device_name(0))
print("vram_gb", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

# Import heavy stack only after restart
import vllm
from orpheus_tts import OrpheusModel

print("vllm", getattr(vllm, "__version__", "?"))
print("OrpheusModel import: OK")


In [ ]:
import json
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)
Path("lab_audio/orpheus_part1").mkdir(parents=True, exist_ok=True)
!ielts-s2a prepare examples/part1_valid.json --pretty -o outputs/part1_manifest.json
m = json.loads(Path("outputs/part1_manifest.json").read_text())
assert m["validation"]["valid"] is True
print("n_requests", len(m["requests"]))
for u in m["prepared_utterances"]:
    if u["event_id"] in {"e004", "e006", "e008", "e011"}:
        print(u["event_id"], u["spoken_text"][:90])


In [ ]:
import json
import os
from pathlib import Path

# Confirm import still works in this session before long render
from orpheus_tts import OrpheusModel
print("preflight OrpheusModel OK")

OUT = Path("lab_audio/orpheus_part1")
REPORT = OUT / "lab_render_report.json"
OUT.mkdir(parents=True, exist_ok=True)

cmd = (
    "python scripts/lab_render_orpheus_from_manifest.py "
    "--manifest outputs/part1_manifest.json "
    "--voice-map configs/voice_maps/orpheus_en_part1.json "
    "--out-dir lab_audio/orpheus_part1 "
    "--temperature 0.6 --repetition-penalty 1.3 --limit 4"
)
print(cmd)
ret = os.system(cmd)
print("exit", ret)
if not REPORT.is_file():
    raise FileNotFoundError("No report — import/render failed. Do not continue tracking.")
report = json.loads(REPORT.read_text())
print("ok", report["ok_count"], "fail", report["fail_count"])
for s in report["segments"]:
    print(s["segment_id"], s["status"], s.get("backend_voice_id"), s.get("error") or "")


In [ ]:
from pathlib import Path
from IPython.display import Audio, display
import json

REPORT_PATH = Path("lab_audio/orpheus_part1/lab_render_report.json")
assert REPORT_PATH.is_file()
report = json.loads(REPORT_PATH.read_text())
segments = report.get("segments") or []
audio_dir = REPORT_PATH.parent
for i, seg in enumerate(segments, 1):
    fname = seg.get("output_filename")
    wav = audio_dir / fname if fname else None
    print("=" * 72)
    print(f"[{i}/{len(segments)}] {seg.get('segment_id')} | {seg.get('speaker_id')} | {seg.get('backend_voice_id')}")
    print("DISPLAY:", seg.get("display_text", ""))
    print("SPOKEN :", seg.get("spoken_text", ""))
    if wav is not None and wav.is_file():
        display(Audio(filename=str(wav)))


In [ ]:
from pathlib import Path
import json
report = json.loads(Path("lab_audio/orpheus_part1/lab_render_report.json").read_text())
reviews = {s["segment_id"]: {"content_match": "", "notes": ""} for s in report.get("segments") or []}
filled = []
for s in report.get("segments") or []:
    h = reviews.get(s["segment_id"]) or {}
    filled.append({
        "segment_id": s.get("segment_id"),
        "display_text": s.get("display_text"),
        "spoken_text": s.get("spoken_text"),
        "backend_voice_id": s.get("backend_voice_id"),
        "output_filename": s.get("output_filename"),
        "content_match": (h.get("content_match") or "").strip().lower(),
        "notes": h.get("notes", ""),
    })
Path("lab_audio/orpheus_part1/segment_review_filled.json").write_text(
    json.dumps(filled, ensure_ascii=False, indent=2) + "\n"
)
print("saved", len(filled), "- edit reviews dict above then re-run to fill matches")


In [ ]:
import json
from pathlib import Path
from IPython.display import Audio, display
!python scripts/lab_concat_segment_wavs.py --report lab_audio/orpheus_part1/lab_render_report.json --out lab_audio/orpheus_part1/part1_full.wav --gap-mode adaptive
full = Path("lab_audio/orpheus_part1/part1_full.wav")
if full.is_file():
    display(Audio(filename=str(full)))


In [ ]:
from pathlib import Path
from IPython.display import Audio, display
import json
from collections import defaultdict

!python scripts/lab_survey_orpheus_voices.py --manifest outputs/part1_manifest.json --inventory configs/voice_maps/orpheus_voice_inventory.json --preset en_shortlist --out-dir lab_audio/orpheus_voice_survey --event-ids e004,e006,e008,e011

rp = Path("lab_audio/orpheus_voice_survey/voice_survey_report.json")
assert rp.is_file()
report = json.loads(rp.read_text())
print("ok/fail", report.get("ok_count"), report.get("fail_count"))
by = defaultdict(list)
for r in report.get("renders") or []:
    by[r.get("event_id")].append(r)
for eid, items in by.items():
    items = sorted(items, key=lambda x: x.get("voice_id") or "")
    print("=" * 72, "LINE", eid)
    if items:
        print("DISPLAY", items[0].get("display_text"))
        print("SPOKEN ", items[0].get("spoken_text"))
    for it in items:
        print("---", it.get("voice_id"), it.get("status"))
        p = Path("lab_audio/orpheus_voice_survey") / it["output_filename"]
        if p.is_file():
            display(Audio(filename=str(p)))


Lab only. not_selected. No silent script edits.
